import sys
import os
import pandas as pd
import py4cytoscape as p4c
import IPython

print("Working directory:", os.getcwd())

In [34]:
import sys
import os
import pandas as pd
import py4cytoscape as p4c
import IPython

print("Working directory:", os.getcwd())

Working directory: /home/jovyan/work/persistent/AOP-RareDiseases-KG/aop-10


In [35]:
print(f"Loading Javascript client ... {p4c.get_browser_client_channel()} on {p4c.get_jupyter_bridge_url()}")
browser_client_js = p4c.get_browser_client_js()
IPython.display.Javascript(browser_client_js)

Loading Javascript client ... 064c8016-8010-4bc4-a182-20ada9947c43 on https://jupyter-bridge.cytoscape.org


<IPython.core.display.Javascript object>

In [36]:
print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()
print("Connected to Cytoscape")

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!
Connected to Cytoscape


In [33]:
print(p4c.cytoscape_version_info())
p4c.cytoscape_ping()

{'apiVersion': 'v1', 'cytoscapeVersion': '3.10.3', 'automationAPIVersion': '1.13.0', 'py4cytoscapeVersion': '1.13.0', 'jupyterBridgeVersion': '0.0.2'}
You are connected to Cytoscape!


'You are connected to Cytoscape!'

In [43]:
import pandas as pd
df_merged = pd.read_csv('aop10_go_genes_with_ke.tsv', sep='\t')
print(df_merged)

     ensembl_gene_id external_gene_name  entrezgene_id       go_id  ke_id
0    ENSG00000281877            RPS6KA1         6195.0  GO:0007268    669
1    ENSG00000291672             PCDHB2        56133.0  GO:0007268    669
2    ENSG00000291676             PCDHB4        56131.0  GO:0007268    669
3    ENSG00000291677             PCDHB5        26167.0  GO:0007268    669
4    ENSG00000291679             PCDHB6        56130.0  GO:0007268    669
..               ...                ...            ...         ...    ...
390  ENSG00000108852               MPP2         4355.0  GO:0060079    682
391  ENSG00000108405              P2RX1         5023.0  GO:0060079    682
392  ENSG00000083454              P2RX5         5026.0  GO:0060079    682
393  ENSG00000115839           RAB3GAP1        22930.0  GO:0060079    682
394              NaN                NaN            NaN         NaN    613

[395 rows x 5 columns]


In [46]:
# Drop rows with missing gene symbol and deduplicate
genes_df = df_merged.dropna(subset=["external_gene_name"]).drop_duplicates(
    subset=["external_gene_name", "ke_id"]
).copy()
 
# ── Build KE node table ───────────────────────────────────────────────────────
ke_nodes = pd.DataFrame([
    {"id": "MIE:667", "label": "MIE:667", "name": "Binding at picrotoxin site, iGABAR chloride channel",               "type": "MIE"},
    {"id": "KE:64",   "label": "KE:64",   "name": "Reduction, Ionotropic GABA receptor Cl- channel conductance",        "type": "KE"},
    {"id": "KE:669",  "label": "KE:669",  "name": "Reduction, Neuronal synaptic inhibition",                            "type": "KE"},
    {"id": "KE:682",  "label": "KE:682",  "name": "Generation, Amplified excitatory postsynaptic potential (EPSP)",     "type": "KE"},
    {"id": "KE:616",  "label": "KE:616",  "name": "Occurrence, A paroxysmal depolarizing shift",                        "type": "KE"},
    {"id": "AO:613",  "label": "AO:613",  "name": "Occurrence, Epileptic seizure",                                      "type": "AO"},
])
 
# KE id mapping
ke_id_map = {667: "MIE:667", 64: "KE:64", 669: "KE:669",
             682: "KE:682", 616: "KE:616", 613: "AO:613"}
 
# ── Build gene node table ─────────────────────────────────────────────────────
gene_nodes = pd.DataFrame({
    "id":             genes_df["external_gene_name"],
    "label":          genes_df["external_gene_name"],
    "name":           genes_df["external_gene_name"],
    "ensembl_id":     genes_df["ensembl_gene_id"],
    "entrez_id":      genes_df["entrezgene_id"].astype(str),
    "go_id":          genes_df["go_id"],
    "type":           "gene",
}).drop_duplicates(subset=["id"]).reset_index(drop=True)
 
# ── Combine all nodes ─────────────────────────────────────────────────────────
all_nodes = pd.concat([ke_nodes, gene_nodes], ignore_index=True)
print(f"Total nodes: {len(all_nodes)} ({len(ke_nodes)} KE + {len(gene_nodes)} gene)")
 
# ── Build KE → KE edges ───────────────────────────────────────────────────────
ke_edges = pd.DataFrame([
    {"source": "MIE:667", "target": "KE:64",  "interaction": "adjacent",    "evidence": "High",     "weight": 3, "edge_type": "KE_KE"},
    {"source": "KE:64",   "target": "KE:669", "interaction": "adjacent",    "evidence": "High",     "weight": 3, "edge_type": "KE_KE"},
    {"source": "KE:669",  "target": "KE:682", "interaction": "adjacent",    "evidence": "High",     "weight": 3, "edge_type": "KE_KE"},
    {"source": "KE:682",  "target": "KE:616", "interaction": "adjacent",    "evidence": "Moderate", "weight": 2, "edge_type": "KE_KE"},
    {"source": "KE:616",  "target": "AO:613", "interaction": "adjacent",    "evidence": "High",     "weight": 3, "edge_type": "KE_KE"}
])
 
# ── Build Gene → KE edges ─────────────────────────────────────────────────────
gene_edges = pd.DataFrame({
    "source":      genes_df["external_gene_name"],
    "target":      genes_df["ke_id"].map(ke_id_map),
    "interaction": "involves_gene",
    "evidence":    "GO",
    "weight":      1,
    "edge_type":   "Gene_KE",
}).dropna(subset=["target"]).drop_duplicates(subset=["source", "target"])
 
all_edges = pd.concat([ke_edges, gene_edges], ignore_index=True)
print(f"Total edges: {len(all_edges)} ({len(ke_edges)} KE-KE + {len(gene_edges)} gene-KE)")
 
# ── Create network in Cytoscape ───────────────────────────────────────────────
suid = p4c.create_network_from_data_frames(
    nodes=all_nodes,
    edges=all_edges,
    source_id_list="source",
    target_id_list="target",
    interaction_type_list="interaction",
    node_id_list="id",
    title="AOP 10 — KE + Gene Network",
    collection="AOP-RareDiseases-KG",
)
print(f"Network created — SUID: {suid}")

Total nodes: 319 (6 KE + 313 gene)
Total edges: 379 (5 KE-KE + 374 gene-KE)
Applying default style...
Applying preferred layout
Network created — SUID: 725387
